# doc2instruct — QLoRA fine-tuning (Qwen 2.5 7B)

Trains one of the models under test in `eval_plan.md` §3. Run this notebook
**once per training-data variant**, changing only `RUN_NAME` and `DATASET_PATHS`:

| Variant | `DATASET_PATHS` | Purpose |
|---|---|---|
| `stage1` | `chatml_dataset.jsonl` | does single-page grounded QA help? |
| `stage1_stage2` | `chatml_dataset.jsonl` + `cross_page_chatml_dataset.jsonl` | does cross-page synthesis add anything on top? |

The base model is evaluated with **no** fine-tuning, so it needs no notebook run.

**This notebook needs a GPU** (~16 GB for 7B QLoRA). It is written for
Kaggle / Colab / RunPod; the repo's own pipeline runs on CPU and does not need one.

### Before you run this
`scripts/check_contamination.py` must exit 0. Fine-tuning on a dataset that
contains held-out papers invalidates every number you produce afterwards.

### Reproducibility
Seed, hyperparameters and dataset fingerprints (SHA256 + record counts) are
written to `run_manifest.json` beside the adapter so a result can always be
traced back to the exact data that produced it.

In [ ]:
# Unsloth gives ~2x faster QLoRA and lower VRAM than vanilla peft for this size.
# On Kaggle/Colab, restart the runtime once after this cell if imports fail.
%pip install -q "unsloth[cu121-torch230] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# ----------------------------------------------------------------------
# CONFIG — the only cell you normally edit.
# ----------------------------------------------------------------------
from pathlib import Path

RUN_NAME = "stage1"  # "stage1" or "stage1_stage2"

# Point these at the files produced by config.train.yaml.
DATA_DIR = Path("output/train_run")
DATASET_PATHS = {
    "stage1": [DATA_DIR / "chatml_dataset.jsonl"],
    "stage1_stage2": [
        DATA_DIR / "chatml_dataset.jsonl",
        DATA_DIR / "cross_page_chatml_dataset.jsonl",
    ],
}[RUN_NAME]

BASE_MODEL = "unsloth/Qwen2.5-7B"  # base, NOT -Instruct: we are the instruction tuning
OUTPUT_DIR = Path(f"outputs/doc2instruct-{RUN_NAME}")

MAX_SEQ_LEN = 2048
SEED = 42

# eval_plan.md §7 open decisions, resolved:
#   r=16 / alpha=32 (alpha = 2r is the common stable default; higher alpha than
#   rank lets a small adapter still move the model on a few-thousand-example set)
#   2 epochs — with ~5-10k examples, 1 epoch underfits and 3+ starts memorising
#   verbatim answers, which inflates our own custom eval. Watch the loss curve
#   and drop to 1 if train loss falls below ~0.5 early.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
EPOCHS = 2
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8  # effective batch 16

print(f"variant : {RUN_NAME}")
print(f"datasets: {[str(p) for p in DATASET_PATHS]}")
print(f"output  : {OUTPUT_DIR}")

## 1. Load and audit the training data

We fingerprint and sanity-check before training, because a silent data problem
here is indistinguishable from a modelling result later.

In [ ]:
import hashlib
import json
from collections import Counter


def load_chatml(paths):
    records, fingerprints = [], {}
    for path in paths:
        if not Path(path).exists():
            raise FileNotFoundError(
                f"{path} not found. Run: python run.py --config config.train.yaml"
            )
        raw = Path(path).read_bytes()
        fingerprints[str(path)] = {
            "sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
        }
        n = 0
        for line in raw.decode("utf-8").splitlines():
            if not line.strip():
                continue
            rec = json.loads(line)
            if not rec.get("messages"):
                continue
            records.append(rec)
            n += 1
        fingerprints[str(path)]["records"] = n
    return records, fingerprints


records, FINGERPRINTS = load_chatml(DATASET_PATHS)
levels = Counter(r.get("metadata", {}).get("record_level", "local") for r in records)
books = {r.get("metadata", {}).get("source_book", "?") for r in records}

print(f"records        : {len(records)}")
print(f"record_level   : {dict(levels)}")
print(f"source papers  : {len(books)}")
print(f"question_type  : {dict(Counter(r.get('metadata', {}).get('question_type', '?') for r in records))}")
for path, fp in FINGERPRINTS.items():
    print(f"  {path}: {fp['records']} records, sha256={fp['sha256'][:16]}...")

# Exact-duplicate guard: identical prompts teach nothing and skew the loss.
seen, dupes = set(), 0
for r in records:
    key = json.dumps(r["messages"], sort_keys=True)
    if key in seen:
        dupes += 1
    seen.add(key)
print(f"exact duplicate records: {dupes}")
assert len(records) > 0, "No training records loaded."

In [ ]:
# Contamination re-check at the point of use. The gate may have been run before
# the dataset was regenerated, so verify against THIS dataset's source papers.
manifest_path = Path("corpus/manifest.jsonl")
if manifest_path.exists():
    holdout = set()
    for line in manifest_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        m = json.loads(line)
        if m.get("split") == "holdout":
            holdout.add(str(m.get("arxiv_id", "")).split("v")[0])
    leaked = {b for b in books if any(h and h in str(b) for h in holdout)}
    if leaked:
        raise SystemExit(
            f"CONTAMINATION: held-out papers present in training data: {leaked}. "
            "Stop and regenerate with config.train.yaml."
        )
    print(f"Contamination check passed: 0 of {len(holdout)} held-out papers present.")
else:
    print("WARNING: corpus/manifest.jsonl not found; cannot verify split isolation.")

## 2. Load the base model in 4-bit and attach LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,       # auto: bf16 where supported
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    # All attention + MLP projections: leaving the MLP out noticeably hurts
    # factual recall, which is exactly what this dataset is teaching.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print(model.print_trainable_parameters())

## 3. Render ChatML into the model's chat template

Loss is computed on the **assistant turn only**. Training on the question text
as well would spend capacity learning to generate questions, which is not the
behaviour being measured.

In [ ]:
from datasets import Dataset

# Qwen2.5 ships a ChatML template, which is the format doc2instruct emits.
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for m in messages %}"
        "{{'<|im_start|>' + m['role'] + '\n' + m['content'] + '<|im_end|>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{'<|im_start|>assistant\n'}}{% endif %}"
    )


def to_text(rec):
    return tokenizer.apply_chat_template(
        rec["messages"], tokenize=False, add_generation_prompt=False
    )


texts = [to_text(r) for r in records]
dataset = Dataset.from_dict({"text": texts}).shuffle(seed=SEED)
print(f"training examples: {len(dataset)}")
print("-" * 70)
print(dataset[0]["text"][:700])

In [ ]:
# Length audit: silently truncated examples lose their answers, which looks
# like a bad model rather than a bad config.
lengths = [len(tokenizer(t).input_ids) for t in texts]
lengths.sort()
over = sum(1 for n in lengths if n > MAX_SEQ_LEN)
print(f"tokens  median={lengths[len(lengths)//2]}  p95={lengths[int(0.95*len(lengths))]}  max={lengths[-1]}")
print(f"examples exceeding MAX_SEQ_LEN ({MAX_SEQ_LEN}): {over} ({100*over/len(lengths):.1f}%)")
if over / len(lengths) > 0.02:
    print("NOTE: consider raising MAX_SEQ_LEN — more than 2% would be truncated.")

## 4. Train

In [ ]:
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,  # keep one QA per example so the loss isn't blurred across items
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        seed=SEED,
        output_dir=str(OUTPUT_DIR / "checkpoints"),
        save_strategy="epoch",
        report_to="none",
    ),
)

train_result = trainer.train()
print(train_result.metrics)

## 5. Save the adapter and a run manifest

In [ ]:
import time

adapter_dir = OUTPUT_DIR / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

manifest = {
    "run_name": RUN_NAME,
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "base_model": BASE_MODEL,
    "datasets": FINGERPRINTS,
    "n_train_examples": len(dataset),
    "hyperparameters": {
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM,
        "max_seq_length": MAX_SEQ_LEN,
        "seed": SEED,
    },
    "train_metrics": getattr(train_result, "metrics", {}),
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(f"adapter  -> {adapter_dir}")
print(f"manifest -> {OUTPUT_DIR / 'run_manifest.json'}")
print(json.dumps(manifest["hyperparameters"], indent=2))

In [ ]:
# Optional: merge to fp16 so lm-eval / vLLM can load it as a normal model.
# Skip if you are happy pointing the eval at base model + adapter.
MERGE = False
if MERGE:
    model.save_pretrained_merged(
        str(OUTPUT_DIR / "merged_16bit"), tokenizer, save_method="merged_16bit"
    )
    print(f"merged -> {OUTPUT_DIR / 'merged_16bit'}")

## 6. Smoke-test the tuned model

A quick qualitative look before spending time on the full eval suite. If the
model rambles or ignores the question, something is wrong with the chat
template rather than the data.

In [ ]:
FastLanguageModel.for_inference(model)

probe = [
    {"role": "system", "content": "You are a helpful tutor grounded in source material."},
    {"role": "user", "content": "What is a Bayes-sufficient representation, and why does it matter for transfer?"},
]
inputs = tokenizer.apply_chat_template(
    probe, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.0, do_sample=False)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Next steps

Download `outputs/doc2instruct-<variant>/` (adapter + `run_manifest.json`), then
back in the repo:

```bash
# Tier 1 + 2 standard benchmarks
python scripts/run_baselines.py \
  --model-args pretrained=Qwen/Qwen2.5-7B,peft=outputs/doc2instruct-stage1/adapter \
  --output-dir eval/baselines/stage1

# Tier 3 custom held-out set
python scripts/run_custom_eval.py \
  --backend hf \
  --model Qwen/Qwen2.5-7B \
  --peft outputs/doc2instruct-stage1/adapter \
  --out eval/custom/results/stage1.json
```

Run the identical commands for the base model and for `stage1_stage2`, then fill
in `eval/RESULTS.md`. Report all three columns even where the result is
unflattering — a Stage 2 that does not help is a finding, not a failure.